# 📂 Customer Support Ticket Dataset Generator

### 👨‍💻 Developed By
**Keshav Sharma**

---

## 📌 Overview

This notebook generates synthetic customer support ticket datasets for an end-to-end Azure Data Engineering pipeline. The generated data simulates a real-world customer support environment and is used for data ingestion, transformation, analytics, and reporting.

---

## 🛠️ Technologies Used

- Python
- Pandas
- Faker
- Random
- Datetime

---

## 📂 Generated Datasets

The notebook generates the following CSV files:

- **Agent Profiles** – Agent details and Team Lead mapping
- **Day 1 Support Tickets** – Customer support tickets for Day 1
- **Day 2 Support Tickets** – Customer support tickets for Day 2

---

## 📋 Features

- Generates realistic synthetic customer support data
- Creates random ticket statuses and resolution times
- Maps agents to Team Leads (TL01–TL08)
- Simulates two days of ticket activity
- Exports clean CSV files for downstream processing
- Provides input datasets for the Customer Support Ticket Resolution Pipeline

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

import random
import string

In [0]:
spark = SparkSession.builder.getOrCreate()

print("Spark Session Started Successfully")

Spark Session Started Successfully


In [0]:
TEAM_LEADS = [
    "TL01",
    "TL02",
    "TL03",
    "TL04",
    "TL05",
    "TL06",
    "TL07",
    "TL08"
]

CATEGORIES = [
    "Technical",
    "Billing",
    "Delivery",
    "Returns",
    "General"
]

STATUS = [
    "Resolved",
    "Open",
    "Pending"
]

In [0]:
agents = []

agent_number = 1

for tl in TEAM_LEADS:
    
    for i in range(6):          # 6 agents per Team Lead
        
        agent_id = f"A{agent_number:03d}"
        
        agent_name = f"Agent_{agent_id}"
        
        role = random.choice([
            "Junior Support Agent",
            "Support Agent",
            "Senior Support Agent"
        ])
        
        agents.append(
            (
                agent_id,
                agent_name,
                role,
                tl
            )
        )
        
        agent_number += 1

In [0]:
agent_schema = StructType([
    
    StructField("agent_id",StringType(),True),
    StructField("agent_name",StringType(),True),
    StructField("role",StringType(),True),
    StructField("team_lead_id",StringType(),True)
    
])

In [0]:
agent_df = spark.createDataFrame(
    agents,
    schema=agent_schema
)

In [0]:
print("Total Agents :",agent_df.count())

display(agent_df)

Total Agents : 48


agent_id,agent_name,role,team_lead_id
A001,Agent_A001,Senior Support Agent,TL01
A002,Agent_A002,Support Agent,TL01
A003,Agent_A003,Senior Support Agent,TL01
A004,Agent_A004,Senior Support Agent,TL01
A005,Agent_A005,Senior Support Agent,TL01
A006,Agent_A006,Support Agent,TL01
A007,Agent_A007,Senior Support Agent,TL02
A008,Agent_A008,Junior Support Agent,TL02
A009,Agent_A009,Junior Support Agent,TL02
A010,Agent_A010,Junior Support Agent,TL02


In [0]:
extra_agents = [

("A049","Agent_A049","Support Agent","TL09"),

("A050","Agent_A050","Support Agent","TL12")

]

extra_df = spark.createDataFrame(
    extra_agents,
    schema=agent_schema
)

agent_df = agent_df.union(extra_df)

In [0]:
print("Final Agent Count :",agent_df.count())

display(agent_df)

Final Agent Count : 50


agent_id,agent_name,role,team_lead_id
A001,Agent_A001,Senior Support Agent,TL01
A002,Agent_A002,Support Agent,TL01
A003,Agent_A003,Senior Support Agent,TL01
A004,Agent_A004,Senior Support Agent,TL01
A005,Agent_A005,Senior Support Agent,TL01
A006,Agent_A006,Support Agent,TL01
A007,Agent_A007,Senior Support Agent,TL02
A008,Agent_A008,Junior Support Agent,TL02
A009,Agent_A009,Junior Support Agent,TL02
A010,Agent_A010,Junior Support Agent,TL02


In [0]:
from random import randint, choice

In [0]:
day1_data = []

ticket_no = 1

agent_ids = [row.agent_id for row in agent_df.collect()]

In [0]:
for i in range(1000):

    ticket_id = f"TKT{ticket_no:05d}"

    agent = choice(agent_ids)

    status = choice([
        "Resolved",
        "Resolved",
        "Resolved",
        "Open",
        "Pending"
    ])

    category = choice(CATEGORIES)

    hours = randint(0,2)
    minutes = randint(0,59)
    seconds = randint(0,59)

    resolution_time = f"{hours}h {minutes}m {seconds}s"

    day1_data.append(

        (
            ticket_id,
            agent,
            status,
            resolution_time,
            category
        )

    )

    ticket_no += 1

In [0]:
day1_data.extend([

("TKT99991",None,"Resolved","0h 25m 12s","Billing"),

("TKT99992","A001","Resolved",None,"Technical"),

("TKT99993","A002","Resolved","BADTIME","General"),

(None,"A003","Resolved","0h 40m 20s","Delivery"),

("TKT99994","A049","Resolved","0h 55m 10s","Returns"),

("TKT99995","A050","Resolved","0h 30m 15s","Technical")

])

In [0]:
ticket_schema = StructType([

    StructField("ticket_id",StringType(),True),

    StructField("agent_id",StringType(),True),

    StructField("status",StringType(),True),

    StructField("resolution_time",StringType(),True),

    StructField("category",StringType(),True)

])

In [0]:
day1_df = spark.createDataFrame(

    day1_data,

    schema=ticket_schema

)

In [0]:
print("Total Day1 Tickets :",day1_df.count())

display(day1_df.limit(200))

Total Day1 Tickets : 1006


ticket_id,agent_id,status,resolution_time,category
TKT00001,A036,Pending,0h 30m 9s,Returns
TKT00002,A042,Resolved,1h 17m 11s,Billing
TKT00003,A004,Resolved,0h 44m 43s,Billing
TKT00004,A005,Pending,0h 28m 27s,General
TKT00005,A039,Resolved,0h 49m 27s,Returns
TKT00006,A011,Resolved,1h 13m 32s,General
TKT00007,A007,Resolved,2h 53m 16s,Returns
TKT00008,A039,Resolved,2h 2m 11s,Returns
TKT00009,A026,Pending,1h 3m 48s,Returns
TKT00010,A021,Resolved,1h 50m 15s,Returns


In [0]:
day2_data = []

ticket_no = 2001

for i in range(800):

    ticket_id = f"TKT{ticket_no:05d}"

    agent = choice(agent_ids)

    status = choice([
        "Resolved",
        "Resolved",
        "Open",
        "Pending"
    ])

    category = choice(CATEGORIES)

    hours = randint(0,2)
    minutes = randint(0,59)
    seconds = randint(0,59)

    resolution_time = f"{hours}h {minutes}m {seconds}s"

    day2_data.append(
        (
            ticket_id,
            agent,
            status,
            resolution_time,
            category
        )
    )

    ticket_no += 1

In [0]:
day2_data.extend([

("TKT29991",None,"Resolved","0h 18m 22s","Billing"),

("TKT29992","A001","Resolved",None,"Technical"),

("TKT29993","A002","Resolved","BADTIME","General"),

(None,"A003","Resolved","0h 42m 10s","Delivery"),

("TKT29994","A049","Resolved","0h 20m 45s","Returns"),

("TKT29995","A050","Resolved","0h 28m 35s","Technical")

])

In [0]:
day2_df = spark.createDataFrame(
    day2_data,
    schema=ticket_schema
)

In [0]:
print("Total Day2 Tickets :", day2_df.count())

display(day2_df.limit(20))

Total Day2 Tickets : 806


ticket_id,agent_id,status,resolution_time,category
TKT02001,A037,Pending,1h 10m 46s,Delivery
TKT02002,A005,Pending,2h 29m 10s,Technical
TKT02003,A012,Resolved,0h 8m 37s,Billing
TKT02004,A004,Resolved,0h 26m 33s,Delivery
TKT02005,A013,Open,1h 20m 8s,Billing
TKT02006,A032,Resolved,2h 22m 15s,General
TKT02007,A027,Open,0h 6m 2s,Billing
TKT02008,A002,Resolved,1h 45m 2s,Technical
TKT02009,A023,Resolved,0h 27m 45s,Technical
TKT02010,A017,Pending,1h 21m 36s,General


In [0]:
agent_pd = agent_df.toPandas()

display(agent_pd)

agent_id,agent_name,role,team_lead_id
A001,Agent_A001,Senior Support Agent,TL01
A002,Agent_A002,Support Agent,TL01
A003,Agent_A003,Senior Support Agent,TL01
A004,Agent_A004,Senior Support Agent,TL01
A005,Agent_A005,Senior Support Agent,TL01
A006,Agent_A006,Support Agent,TL01
A007,Agent_A007,Senior Support Agent,TL02
A008,Agent_A008,Junior Support Agent,TL02
A009,Agent_A009,Junior Support Agent,TL02
A010,Agent_A010,Junior Support Agent,TL02


In [0]:
day2_pd = day2_df.toPandas()

display(day2_pd)

ticket_id,agent_id,status,resolution_time,category
TKT02001,A037,Pending,1h 10m 46s,Delivery
TKT02002,A005,Pending,2h 29m 10s,Technical
TKT02003,A012,Resolved,0h 8m 37s,Billing
TKT02004,A004,Resolved,0h 26m 33s,Delivery
TKT02005,A013,Open,1h 20m 8s,Billing
TKT02006,A032,Resolved,2h 22m 15s,General
TKT02007,A027,Open,0h 6m 2s,Billing
TKT02008,A002,Resolved,1h 45m 2s,Technical
TKT02009,A023,Resolved,0h 27m 45s,Technical
TKT02010,A017,Pending,1h 21m 36s,General


In [0]:
agent_df.toPandas().to_csv("agent_profiles.csv", index=False)

day1_df.toPandas().to_csv("day1_tickets.csv", index=False)

day2_df.toPandas().to_csv("day2_tickets.csv", index=False)

print("Done")

Done


In [0]:
day1_pd = day1_df.toPandas()

display(day1_pd)

ticket_id,agent_id,status,resolution_time,category
TKT00001,A036,Pending,0h 30m 9s,Returns
TKT00002,A042,Resolved,1h 17m 11s,Billing
TKT00003,A004,Resolved,0h 44m 43s,Billing
TKT00004,A005,Pending,0h 28m 27s,General
TKT00005,A039,Resolved,0h 49m 27s,Returns
TKT00006,A011,Resolved,1h 13m 32s,General
TKT00007,A007,Resolved,2h 53m 16s,Returns
TKT00008,A039,Resolved,2h 2m 11s,Returns
TKT00009,A026,Pending,1h 3m 48s,Returns
TKT00010,A021,Resolved,1h 50m 15s,Returns
